# Can we predict next-month arrears?
Target: `arrears_next_month`. Methodology follows `checklist.md`.

Key decisions up front:
- **`final_notice_sent` is dropped.** columns.csv says it is recorded in the billing cycle *after* the billing month, so it is not known at prediction time (it nearly restates the outcome: ~87% arrears when 1 vs ~1% when 0). `meter_ref` is an identifier and is used only for grouping, never as a feature.
- The model is meant for future months, so the test set is the **latest rows**; accounts repeat across rows, so the test set also holds **whole accounts** that never appear in training.

In [ ]:
import pandas as pd, numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.base import clone
df=pd.read_csv('data.csv',parse_dates=['billing_month'])
print(df.shape, df.meter_ref.nunique(),'accounts', df.billing_month.min().date(), df.billing_month.max().date())
print(df.groupby('billing_month').arrears_next_month.mean().round(3).to_string())

The arrears rate rises from ~2-3% to ~26% over the period: strong drift. Performance on the latest months is therefore the honest estimate for future use, and the base rate in test will differ from training.

## 1. Test set (set aside before any modelling)
The last 4 billing months (2025-09 to 2025-12) form the test period. A random 20% of accounts are test accounts; the test set is their rows in the test period. Training rows are all rows before the test period from the *other* accounts (earlier rows of test accounts are discarded, so no account is in both).

In [ ]:
rng=np.random.RandomState(0)
accounts=df.meter_ref.unique()
test_accts=set(rng.choice(accounts,size=int(0.2*len(accounts)),replace=False))
cutoff=pd.Timestamp('2025-09-01')
is_test_acct=df.meter_ref.isin(test_accts)
test=df[is_test_acct&(df.billing_month>=cutoff)].copy()
train=df[~is_test_acct&(df.billing_month<cutoff)].copy().sort_values('billing_month').reset_index(drop=True)
assert not set(train.meter_ref)&set(test.meter_ref)
assert train.billing_month.max()<test.billing_month.min()
print(len(train),len(test),'rows (train, test); train rate %.3f'%train.arrears_next_month.mean())

## 7. Features (known at prediction time)
`tariff`, `occupants`, `kwh`, `bill_eur`, `late_payments_12m` describe the billing month or the 12 months up to it. `billing_month` is not used as a feature (a calendar date cannot extrapolate into future months under drift). Dropped: `meter_ref` (identifier), `final_notice_sent` (recorded after the billing month).

In [ ]:
FEATS=['tariff','occupants','kwh','bill_eur','late_payments_12m']
CAT=['tariff']; NUM=[f for f in FEATS if f not in CAT]
TARGET='arrears_next_month'
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])

## 4. Candidate models (named before any scoring)
1. `baseline`: predicts the training prevalence for everyone (trivial).
2. `logreg_C0.1`, `logreg_C1`, `logreg_C10`: logistic regression (scaled numerics, one-hot tariff).
3. `hgb_shallow`: HistGradientBoosting, max_depth=3, 100 iters, lr 0.05.
4. `hgb_deep`: HistGradientBoosting, max_depth=6, 200 iters, lr 0.05.

Selection metric: mean CV average precision (PR-AUC; the positive class is rare and its rate changes). The decision threshold is chosen on pooled CV out-of-fold predictions to maximise F1.

In [ ]:
candidates={
 'baseline':Pipeline([('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1':Pipeline([('p',prep(True)),('m',LogisticRegression(C=1,max_iter=1000))]),
 'logreg_C10':Pipeline([('p',prep(True)),('m',LogisticRegression(C=10,max_iter=1000))]),
 'hgb_shallow':Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,max_iter=100,learning_rate=0.05,random_state=0))]),
 'hgb_deep':Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=6,max_iter=200,learning_rate=0.05,random_state=0))]),
}
list(candidates)

## 2-3. Cross-validation inside training rows only
Forward-chaining folds: validation = a block of 3 consecutive months; training = all earlier months. Validation accounts (a random ~20% of training accounts per fold) are also removed from that fold's training rows, so folds are both later-in-time and whole-entity. Preprocessing lives inside each `Pipeline`, which is cloned and fit on the fold's training rows only.

In [ ]:
months=sorted(train.billing_month.unique())
val_starts=months[-15::3]            # 5 folds of 3 months each
folds=[]
tr_accts=train.meter_ref.unique()
for i,s in enumerate(val_starts):
    s=pd.Timestamp(s); e=s+pd.DateOffset(months=3)
    r=np.random.RandomState(100+i)
    va_accts=set(r.choice(tr_accts,size=int(0.2*len(tr_accts)),replace=False))
    va=train[train.meter_ref.isin(va_accts)&(train.billing_month>=s)&(train.billing_month<e)]
    tr=train[~train.meter_ref.isin(va_accts)&(train.billing_month<s)]
    assert not set(tr.meter_ref)&set(va.meter_ref) and tr.billing_month.max()<va.billing_month.min()
    folds.append((tr.index.values,va.index.values))
    print(s.date(),len(tr),len(va),'val rate %.3f'%va[TARGET].mean())
oof={n:[] for n in candidates}; scores={n:[] for n in candidates}; aucs={n:[] for n in candidates}; ys=[]
for tri,vai in folds:
    ytr,yva=train.loc[tri,TARGET],train.loc[vai,TARGET]; ys.append(yva.values)
    for n,m in candidates.items():
        mm=clone(m).fit(train.loc[tri,FEATS],ytr)
        p=mm.predict_proba(train.loc[vai,FEATS])[:,1]; oof[n].append(p)
        scores[n].append(average_precision_score(yva,p)); aucs[n].append(roc_auc_score(yva,p))
cv=pd.DataFrame({n:{'mean_AP':np.mean(scores[n]),'sd_AP':np.std(scores[n]),'mean_AUC':np.mean(aucs[n])} for n in scores}).T
cv.round(3)

## 5. Choice on CV only (threshold from pooled out-of-fold predictions)

In [ ]:
best=cv.drop('baseline').mean_AP.idxmax()
yall=np.concatenate(ys); pall=np.concatenate(oof[best])
ths=np.linspace(0.02,0.9,89)
f1s=[f1_score(yall,pall>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| CV threshold:',round(thr,3),'| CV F1 at threshold: %.3f'%max(f1s))
print('CV AP of baseline %.3f vs chosen %.3f'%(cv.loc['baseline','mean_AP'],cv.loc[best,'mean_AP']))

**Decision (recorded before the test set is scored):** the model and threshold printed above are final, with the features listed in section 7 and the hyperparameters of the named candidate. The test set has not been touched so far.

## 6. Refit on all training rows; score the test set once

In [ ]:
final=clone(candidates[best]).fit(train[FEATS],train[TARGET])
p=final.predict_proba(test[FEATS])[:,1]; y=test[TARGET].values; pred=p>=thr
base=clone(candidates['baseline']).fit(train[FEATS],train[TARGET]).predict_proba(test[FEATS])[:,1]
res=pd.Series({'test_rows':len(test),'test_prevalence':y.mean(),'AP (headline)':average_precision_score(y,p),
 'AP baseline':average_precision_score(y,base),'ROC AUC':roc_auc_score(y,p),
 'precision@thr':precision_score(y,pred),'recall@thr':recall_score(y,pred),'F1@thr':f1_score(y,pred)})
res.round(3)

## Conclusion
See the test table above (scored once, no re-tuning). Notes:
- Compare AP with the test prevalence: the no-skill AP equals the prevalence, which is much higher in test than in training because arrears rise over time, so a threshold tuned on earlier months may under-flag later ones.
- Excluding `final_notice_sent` is deliberate: it would inflate performance with information that does not exist when the prediction is needed.
- The test set is 4 months of ~275 accounts; estimates carry sampling noise and an account's repeated rows are correlated.